In [1]:
import pandas as pd
import plotly.express as px
from pathlib import Path


In [2]:
DATA_PATH = Path("../vehicles_us.csv")
# Verifica se o arquivo existe e imprime o caminho absoluto
print("Caminho do arquivo:", DATA_PATH.resolve())
print("Arquivo encontrado:", DATA_PATH.exists())

Caminho do arquivo: D:\Guilherme\Personal\MBA\Sprint 4\Final project\Sprint-4-Project\vehicles_us.csv
Arquivo encontrado: True


In [3]:
car_data = pd.read_csv(DATA_PATH)
# Exibe informações sobre o DataFrame, quantidade de anúncios e colunas
print(f"Quantidade de anúncios: {car_data.shape[0]:,}")
print(f"Quantidade de colunas: {car_data.shape[1]}")

Quantidade de anúncios: 51,525
Quantidade de colunas: 13


In [4]:
car_data.head()
#observando os tipos de dados e colunas presentes

,price,model_year,model,condition,cylinders,fuel,odometer,transmission,type,paint_color,is_4wd,date_posted,days_listed
0,9400,2011.0,bmw x5,good,6.0,gas,145000.0,automatic,SUV,NaN,1.0,2018-06-23,19
1,25500,NaN,ford f-150,good,6.0,gas,88705.0,automatic,pickup,white,1.0,2018-10-19,50
2,5500,2013.0,hyundai sonata,like new,4.0,gas,110000.0,automatic,sedan,red,NaN,2019-02-07,79
3,1500,2003.0,ford f-150,fair,8.0,gas,NaN,automatic,pickup,NaN,NaN,2019-03-22,9
4,14900,2017.0,chrysler 200,excellent,4.0,gas,80903.0,automatic,sedan,black,NaN,2019-04-02,28


In [5]:
# Observo as informações do DataFrame, incluindo tipos de dados e valores ausentes por coluna
car_data.info()
print("\n\n")
missing_values = car_data.isna().sum()

print("Valores ausentes por coluna:")
print(missing_values[missing_values > 0].sort_values(ascending=False))

<class 'pandas.DataFrame'>
RangeIndex: 51525 entries, 0 to 51524
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   price         51525 non-null  int64  
 1   model_year    47906 non-null  float64
 2   model         51525 non-null  str    
 3   condition     51525 non-null  str    
 4   cylinders     46265 non-null  float64
 5   fuel          51525 non-null  str    
 6   odometer      43633 non-null  float64
 7   transmission  51525 non-null  str    
 8   type          51525 non-null  str    
 9   paint_color   42258 non-null  str    
 10  is_4wd        25572 non-null  float64
 11  date_posted   51525 non-null  str    
 12  days_listed   51525 non-null  int64  
dtypes: float64(4), int64(2), str(7)
memory usage: 7.7 MB



Valores ausentes por coluna:
is_4wd         25953
paint_color     9267
odometer        7892
cylinders       5260
model_year      3619
dtype: int64


In [6]:
car_data.describe()
# aqui eu posso ver as estatísticas descritivas das colunas numéricas, como média, mediana, mínimo, máximo e quartis.
# Isso me ajuda a entender a distribuição dos dados e identificar possíveis outliers.

,price,model_year,cylinders,odometer,is_4wd,days_listed
count,51525.000000,47906.000000,46265.000000,43633.000000,25572.0,51525.00000
mean,12132.464920,2009.750470,6.125235,115553.461738,1.0,39.55476
std,10040.803015,6.282065,1.660360,65094.611341,0.0,28.20427
min,1.000000,1908.000000,3.000000,0.000000,1.0,0.00000
25%,5000.000000,2006.000000,4.000000,70000.000000,1.0,19.00000
50%,9000.000000,2011.000000,6.000000,113000.000000,1.0,33.00000
75%,16839.000000,2014.000000,8.000000,155000.000000,1.0,53.00000
max,375000.000000,2019.000000,12.000000,990000.000000,1.0,271.00000


Os resultados mostram que a leitura dos dados está correta e já identificamos pontos importantes para a análise exploratória. Antes de construir o aplicativo, vamos investigar os dados e criar visualizações que realmente ajudem a compreender o mercado de veículos usados.
A diferença entre a média (mean: US$12.132) e a mediana (median: US$ 9.000) sugere que os anúncios de veículos mais caros elevam o preço médio.
Os dados ausentes devem ser tratados com cautela, em especial, is_4wd que indica um valor binário, que identifica o carro 4x4 ou não.
Os valores estatísticos de model_year, cylinders e odometer (quilometragem) são compatíveis com os dados analisados.

Vou manter os dados originais em car_data e fazer a análise em células separadas. Assim, evito modificar os dados antes de entender suas características.

In [7]:
car_data["date_posted"] = pd.to_datetime(car_data["date_posted"])

print("Tipo de date_posted:", car_data["date_posted"].dtype)

print("\nTipos de dados:")
print(car_data.dtypes)

Tipo de date_posted: datetime64[us]

Tipos de dados:
price                    int64
model_year             float64
model                      str
condition                  str
cylinders              float64
fuel                       str
odometer               float64
transmission               str
type                       str
paint_color                str
is_4wd                 float64
date_posted     datetime64[us]
days_listed              int64
dtype: object


Preciso investigar os outliers de preço, pois um carro listado a US$1 é suspeito, US$375.000 também não é comum. Podem ser registros atípicos ou casos legítimos.

In [10]:
fig_price = px.histogram(
    car_data,
    x="price",
    nbins=600,
    title="Distribuição dos preços dos veículos",
    labels={
        "price": "Preço (US$)",
        "count": "Quantidade de anúncios"
    }
)

fig_price.show()

Observo 11 valores acima de US$140k, que não parece equivocado.
Ainda não consigo concluir sobre os valores inferiores, vou verificar contra a quilometragem.

In [11]:
fig_odometer = px.scatter(
    car_data,
    x="odometer",
    y="price",
    title="Relação entre quilometragem e preço",
    labels={
        "odometer": "Quilometragem (milhas)",
        "price": "Preço (US$)"
    },
    opacity=0.35
)

fig_odometer.show()

É possível observar que os preços tendem a baixar conforme a quilometragem aumenta. Ainda preciso investigar mais sobre os preços baixos. Vou criar uma visualização do intervalor 0-50k

In [14]:
fig_price_zoom = px.histogram(
    car_data[car_data["price"] <= 50000],
    x="price",
    nbins=600,
    title="Distribuição dos preços até US$ 50.000",
    labels={
        "price": "Preço (US$)",
        "count": "Quantidade de anúncios"
    }
)

fig_price_zoom.show()

Observo 863 valores entre US$0 e US$99

In [12]:
car_data.nsmallest(20, "price")[
    ["price", "model_year", "model", "condition", "odometer", "type"]
]

,price,model_year,model,condition,odometer,type
405,1,2014.0,chevrolet camaro,excellent,71310.0,coupe
3063,1,1998.0,chevrolet silverado,good,164000.0,pickup
3808,1,2007.0,chevrolet tahoe,good,200.0,SUV
3902,1,1996.0,ford f-150,fair,163000.0,truck
4140,1,2004.0,chevrolet silverado,excellent,83000.0,pickup
5612,1,2006.0,gmc sierra,excellent,192960.0,truck
5700,1,2015.0,ram 2500,excellent,103549.0,truck
5718,1,NaN,toyota tacoma,good,168955.0,truck
5907,1,2011.0,toyota tacoma,good,168955.0,truck
6012,1,2015.0,ram 2500,excellent,103549.0,truck


In [16]:
# com isso terei números objetivos para descrever o conjunto de dados.
price_limits = [1000, 3000, 5000, 10000]

for limit in price_limits:
    quantity = (car_data["price"] < limit).sum()
    percentage = quantity / len(car_data) * 100

    print(
        f"Abaixo de US$ {limit:,.0f}: "
        f"{quantity:,} anúncios ({percentage:.1f}%)"
    )

Abaixo de US$ 1,000: 1,404 anúncios (2.7%)
Abaixo de US$ 3,000: 5,349 anúncios (10.4%)
Abaixo de US$ 5,000: 12,834 anúncios (24.9%)
Abaixo de US$ 10,000: 28,045 anúncios (54.4%)


In [17]:
low_price = car_data[car_data["price"] < 1000]

print("Anúncios abaixo de US$ 1.000:", len(low_price))
print("\nCondição dos veículos:")
print(low_price["condition"].value_counts(dropna=False))

print("\nAno dos veículos:")
print(low_price["model_year"].describe())

print("\nModelos mais frequentes:")
print(low_price["model"].value_counts().head(10))

Anúncios abaixo de US$ 1.000: 1404

Condição dos veículos:
condition
excellent    983
fair         185
good         106
like new      95
salvage       28
new            7
Name: count, dtype: int64

Ano dos veículos:
count    1314.000000
mean     2011.912481
std         7.506531
min      1966.000000
25%      2007.000000
50%      2015.000000
75%      2018.000000
max      2019.000000
Name: model_year, dtype: float64

Modelos mais frequentes:
model
ford f-150             80
dodge charger          79
ram 3500               76
jeep wrangler          64
jeep grand cherokee    57
chevrolet camaro       54
ram 1500               51
toyota tundra          47
chevrolet tahoe        40
honda civic            38
Name: count, dtype: int64


Esses resultados revelam algo importante: não posso assumir que os anúncios abaixo de US$ 1.000 sejam apenas veículos velhos ou em más condições.
Há indícios de que preciso investigar a consistência dos dados antes de decidir como tratá-los.
- 1.404 anúncios têm preço inferior a US$ 1.000, correspondendo a 2,7% da base.
- Entre eles, 983 estão classificados como excellent — aproximadamente 70% dos anúncios baratos.
- O ano mediano é 2015, portanto boa parte desses veículos não é particularmente antiga.
- Há registros com quilometragem ausente e possíveis inconsistências que merecem investigação.
Isso não prova que os preços estejam errados. Pode haver veículos anunciados com valores promocionais, entradas, erros de cadastro ou outras particularidades. Preciso examinar os registros antes de concluir.

In [19]:
print("Menor preço:", car_data["price"].min())

for limite in [1, 10, 50, 100, 500, 999]:
    quantidade = (car_data["price"] <= limite).sum()
    percentual = quantidade / len(car_data) * 100

    print(
        f"Preço <= US$ {limite:,}: "
        f"{quantidade:,} anúncios ({percentual:.2f}%)"
    )

Menor preço: 1
Preço <= US$ 1: 798 anúncios (1.55%)
Preço <= US$ 10: 803 anúncios (1.56%)
Preço <= US$ 50: 822 anúncios (1.60%)
Preço <= US$ 100: 863 anúncios (1.67%)
Preço <= US$ 500: 1,148 anúncios (2.23%)
Preço <= US$ 999: 1,404 anúncios (2.72%)


Há três observações importantes:
- 798 anúncios, ou 1,55% da base, têm preço exatamente igual a US$ 1, pois esse é o menor preço possível no conjunto.
- Apenas 24 anúncios adicionais separam o limite de US$ 1 do limite de US$ 50.
- A quantidade cresce mais significativamente entre US$ 100 e US$ 999, indicando que os preços baixos não se concentram exclusivamente nos valores simbólicos.
Os 798 anúncios de US$ 1 merecem atenção especial. Entretanto, ainda não posso afirmar que sejam erros: preciso examinar os registros e, se possível, entender como o conjunto de dados foi produzido.

Próximo passo será investigar os anúncios de US$1 a fim de que o critério decisivo em manter ou excluir a entrada seja baseado em evidências.

In [20]:
price_one = car_data[car_data["price"] == 1]

print("Quantidade de anúncios por US$ 1:", len(price_one))

print("\nCondição:")
print(price_one["condition"].value_counts(dropna=False))

print("\nAno do veículo:")
print(price_one["model_year"].describe())

print("\nModelos mais frequentes:")
print(price_one["model"].value_counts().head(15))

print("\nAmostra dos anúncios:")
print(
    price_one[
        ["price", "model_year", "model", "condition", "odometer", "type"]
    ].head(20).to_string(index=False)
)

Quantidade de anúncios por US$ 1: 798

Condição:
condition
excellent    741
like new      27
good          18
fair           9
new            3
Name: count, dtype: int64

Ano do veículo:
count     746.000000
mean     2015.621984
std         4.486131
min      1966.000000
25%      2015.000000
50%      2017.000000
75%      2018.000000
max      2019.000000
Name: model_year, dtype: float64

Modelos mais frequentes:
model
ram 3500               76
dodge charger          76
ford f-150             60
jeep wrangler          55
chevrolet camaro       51
toyota tundra          46
chevrolet tahoe        37
nissan rogue           35
ram 1500               34
jeep grand cherokee    31
ford mustang           21
honda civic            20
gmc sierra 1500        17
gmc acadia             15
chevrolet suburban     15
Name: count, dtype: int64

Amostra dos anúncios:
 price  model_year               model condition  odometer     type
     1      2014.0    chevrolet camaro excellent   71310.0    coupe
     

Os 798 anúncios com preço de US$ 1 parecem formar um padrão sistemático de dados, e não apenas uma coleção de veículos baratos.
- 741 dos 798 anúncios (92,9%) estão classificados como excellent.
- O ano mediano é 2017, relativamente recente.
- Há modelos como Ram 3500, Dodge Charger e Ford F-150 entre os mais frequentes.
- Os anúncios incluem veículos com quilometragens variadas, e alguns apresentam dados ausentes.
Essa combinação torna pouco plausível interpretar todos esses registros literalmente como vendas por US$ 1. Ainda assim, não sei se são erros, valores substitutos ou outra convenção da fonte.

In [21]:
# Aqui comparo os anúncios com preço de US$ 1 com os demais anúncios, para entender melhor a distribuição dos dados e 
# identificar possíveis padrões ou anomalias.

price_one = car_data[car_data["price"] == 1]
other_prices = car_data[car_data["price"] != 1]

print("Anúncios com preço igual a US$ 1:", len(price_one))
print("Anúncios com outros preços:", len(other_prices))

print("\nDistribuição por tipo — preço de US$ 1:")
print(price_one["type"].value_counts().head(10))

print("\nDistribuição por tipo — demais preços:")
print(other_prices["type"].value_counts().head(10))

print("\nDistribuição por condição — demais preços:")
print(other_prices["condition"].value_counts())

Anúncios com preço igual a US$ 1: 798
Anúncios com outros preços: 50727

Distribuição por tipo — preço de US$ 1:
type
truck          287
SUV            250
sedan          141
coupe           74
van             16
convertible     11
hatchback        9
pickup           7
mini-van         2
other            1
Name: count, dtype: int64

Distribuição por tipo — demais preços:
type
SUV            12155
truck          12066
sedan          12013
pickup          6981
coupe           2229
wagon           1541
mini-van        1159
hatchback       1038
van              617
convertible      435
Name: count, dtype: int64

Distribuição por condição — demais preços:
condition
excellent    24032
good         20127
like new      4715
fair          1598
new            140
salvage        115
Name: count, dtype: int64


O preço é suspeito, mas não tenho evidência suficiente para afirmar que são erros. Vou manter os dados originais e deixar a análise transparente, sem excluir anúncios arbitrariamente.
Em seguida, verificarei duplicatas completas.

In [22]:
print("Linhas duplicadas completas:", car_data.duplicated().sum())

print(
    "Duplicatas completas entre anúncios de US$ 1:",
    price_one.duplicated().sum()
)

Linhas duplicadas completas: 0
Duplicatas completas entre anúncios de US$ 1: 0


Nenhuma duplicata

In [23]:


print("RESUMO FINAL DA BASE")
print("=" * 50)
print(f"Total de anúncios: {len(car_data):,}")
print(f"Total de colunas: {len(car_data.columns)}")
print(f"Preços iguais a US$ 1: {(car_data['price'] == 1).sum():,}")
print(f"Linhas duplicadas completas: {car_data.duplicated().sum():,}")

print("\nVALORES AUSENTES")
print(car_data.isna().sum().sort_values(ascending=False))

print("\nESTATÍSTICAS DE PREÇO")
print(car_data["price"].describe())

print("\nPREÇOS ABAIXO DE US$ 1.000")
print(f"{(car_data['price'] < 1000).sum():,} anúncios")

RESUMO FINAL DA BASE
Total de anúncios: 51,525
Total de colunas: 13
Preços iguais a US$ 1: 798
Linhas duplicadas completas: 0

VALORES AUSENTES
is_4wd          25953
paint_color      9267
odometer         7892
cylinders        5260
model_year       3619
condition           0
model               0
price               0
fuel                0
type                0
transmission        0
date_posted         0
days_listed         0
dtype: int64

ESTATÍSTICAS DE PREÇO
count     51525.000000
mean      12132.464920
std       10040.803015
min           1.000000
25%        5000.000000
50%        9000.000000
75%       16839.000000
max      375000.000000
Name: price, dtype: float64

PREÇOS ABAIXO DE US$ 1.000
1,404 anúncios
